# SRBCT — Baseline Clustering Comparison

## 1. Objective

In this notebook, we evaluate baseline clustering methods on the SRBCT dataset and compare their performance with IF-PCA, i-IF-PCA, and i-IF-Lap.

The goal is to understand whether the iterative feature-selection approach proposed in the i-IF-Learn paper provides improvements over traditional clustering, dimensionality reduction, and deep-learning-based methods.

We will evaluate clustering performance using Accuracy (ACC) and Adjusted Rand Index (ARI), along with execution time. All methods will use the same dataset, and the true class labels will only be used for evaluation.

We will begin with K-Means and UMAP, followed by IF-VAE, DEC, SC-FS, and other comparisons where feasible. We will also investigate whether features selected by i-IF-Lap improve UMAP clustering performance.

In [7]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import time

from sklearn.cluster import KMeans
from sklearn.preprocessing import StandardScaler

from Src.evaluation import evaluate_clustering
from Src.data_loader import load_dataset
print("Basic imports successful!")


Basic imports successful!


In [8]:
# Load SRBCT using the existing dataset loader
X, y, K = load_dataset(
    "srbct",
    microarray_dir="./10 Gene Microarray Datasets/10 Gene Microarray Datasets"
)

# Verify dataset dimensions and class labels
print("Dataset: SRBCT")
print("X shape:", X.shape)
print("y shape:", y.shape)
print("Number of clusters (K):", K)
print("Unique labels:", np.unique(y))
print("Class distribution:", np.unique(y, return_counts=True)[1])

Dataset: SRBCT
X shape: (63, 2308)
y shape: (63,)
Number of clusters (K): 4
Unique labels: [0 1 2 3]
Class distribution: [23 20 12  8]


## 2. K-Means Baseline

I first apply K-Means directly to the SRBCT dataset using all 2,308 features, without additional scaling or feature selection. The model uses four clusters, 30 initializations, and a fixed random seed of 42.

Clustering performance is evaluated using ACC and ARI, and execution time is recorded for comparison with the other methods.
**Experimental Settings:** K-Means was run with \(K=4\), corresponding to the number of classes in SRBCT, using 30 initializations and a fixed random seed of 42. All 2,308 features were used without additional scaling. True class labels were used only to calculate ACC and ARI.

In [9]:

import time

from Src.baseline import run_kmeans
from Src.evaluation import evaluate_clustering

# Run K-Means on the original SRBCT data (no scaling)
start_time = time.perf_counter()

kmeans_result = run_kmeans(
    X,
    K,
    random_state=42,
    n_init=30
)

kmeans_runtime = time.perf_counter() - start_time

# Evaluate clustering performance
kmeans_metrics = evaluate_clustering(
    y,
    kmeans_result["labels"]
)

print("K-Means ACC:", round(kmeans_metrics["Accuracy"], 4))
print("K-Means ARI:", round(kmeans_metrics["ARI"], 4))
print("Runtime (s):", round(kmeans_runtime, 4))


K-Means ACC: 0.5397
K-Means ARI: 0.1283
Runtime (s): 1.7684


C:\Users\priyam\Python\Lib\site-packages\sklearn\cluster\_kmeans.py:1419: UserWarning: KMeans is known to have a memory leak on Windows with MKL, when there are less chunks than available threads. You can avoid it by setting the environment variable OMP_NUM_THREADS=1.
  warnings.warn(


In [10]:

# K-Means preprocessing sensitivity: standardized features

from sklearn.preprocessing import StandardScaler

X_scaled = StandardScaler().fit_transform(X)
X_scaled *= np.sqrt((X.shape[0] - 1) / X.shape[0])

start_time = time.perf_counter()

kmeans_scaled_result = run_kmeans(
    X_scaled,
    K,
    random_state=42,
    n_init=30
)

kmeans_scaled_runtime = time.perf_counter() - start_time

kmeans_scaled_metrics = evaluate_clustering(
    y,
    kmeans_scaled_result["labels"]
)

print("Scaled K-Means ACC:", round(kmeans_scaled_metrics["Accuracy"], 4))
print("Scaled K-Means ARI:", round(kmeans_scaled_metrics["ARI"], 4))
print("Runtime (s):", round(kmeans_scaled_runtime, 4))


C:\Users\priyam\Python\Lib\site-packages\sklearn\cluster\_kmeans.py:1419: UserWarning: KMeans is known to have a memory leak on Windows with MKL, when there are less chunks than available threads. You can avoid it by setting the environment variable OMP_NUM_THREADS=1.
  warnings.warn(


Scaled K-Means ACC: 0.5714
Scaled K-Means ARI: 0.2101
Runtime (s): 0.1224


### K-Means: Preprocessing Sensitivity and Comparison with the Paper

I evaluated K-Means on SRBCT using two preprocessing configurations: original features without additional scaling and standardized features. The number of clusters (K=4), random seed (42), and number of initializations (30) were kept constant.

Without scaling, K-Means achieved an ACC of 0.5397 and ARI of 0.1283, compared with the paper's reported values of 0.556 and 0.121. After standardization, ACC improved to 0.5714 and ARI to 0.2101.

These results show that preprocessing can influence clustering performance. Although standardization produced better results in this experiment, this is treated as a preprocessing sensitivity analysis rather than selecting the best configuration solely using ground-truth labels.

## 3. SpecGEM Baseline

SpecGEM is a PCA-based clustering method that uses singular value decomposition (SVD) to obtain a lower-dimensional representation before applying K-Means.

I evaluate SpecGEM on SRBCT to compare conventional dimensionality reduction with the feature-selection approach used in IF-PCA and i-IF-Learn.

The implementation standardizes the input features, retains \(K-1=3\) components, and applies K-Means with 30 initializations and a fixed seed of 42.

In [11]:

from Src.baseline import run_specgem
import time

# SpecGEM performs its own feature standardization
start_time = time.perf_counter()

specgem_labels = run_specgem(
    X,
    K,
    seed=42
)

specgem_runtime = time.perf_counter() - start_time

specgem_metrics = evaluate_clustering(
    y,
    specgem_labels
)

print("SpecGEM ACC:", round(specgem_metrics["Accuracy"], 4))
print("SpecGEM ARI:", round(specgem_metrics["ARI"], 4))
print("Runtime (s):", round(specgem_runtime, 4))


C:\Users\priyam\Python\Lib\site-packages\sklearn\cluster\_kmeans.py:1419: UserWarning: KMeans is known to have a memory leak on Windows with MKL, when there are less chunks than available threads. You can avoid it by setting the environment variable OMP_NUM_THREADS=1.
  warnings.warn(


SpecGEM ACC: 0.4762
SpecGEM ARI: 0.0975
Runtime (s): 0.1001


### SpecGEM Results

SpecGEM achieved an ACC of 0.4762 and ARI of 0.0975 on SRBCT, compared with the paper's reported values of 0.492 and 0.125. The runtime was approximately 0.13 seconds.

The results also match my earlier implementation, confirming consistency across the two notebooks.

SpecGEM performed worse than standalone K-Means in this experiment. This suggests that reducing the data to three principal components did not improve clustering performance for SRBCT. One possible explanation is that the reduced representation does not retain all the information needed to distinguish the four classes.

## 4. UMAP + K-Means Baseline

UMAP is a nonlinear dimensionality reduction technique that aims to preserve the underlying structure of high-dimensional data.

I evaluate UMAP followed by K-Means on SRBCT to compare nonlinear dimensionality reduction against the PCA-based and iterative feature-selection methods.

The implementation uses cosine distance, 8 nearest neighbors, and \(K+2=6\) embedding dimensions. K-Means is then applied with four clusters, 30 initializations, and a fixed seed of 42.

Clustering performance is evaluated using ACC and ARI, along with execution time.

In [12]:

from Src.baseline import run_umap
import time

# Run UMAP + K-Means using the original implementation settings
start_time = time.perf_counter()

umap_result = run_umap(
    X,
    K,
    seed=42
)

umap_runtime = time.perf_counter() - start_time

umap_metrics = evaluate_clustering(
    y,
    umap_result["labels"]
)

print("UMAP ACC:", round(umap_metrics["Accuracy"], 4))
print("UMAP ARI:", round(umap_metrics["ARI"], 4))
print("Runtime (s):", round(umap_runtime, 4))
print("Embedding shape:", umap_result["embedding"].shape)


C:\Users\priyam\Python\Lib\site-packages\umap\umap_.py:1952: UserWarning: n_jobs value 1 overridden to 1 by setting random_state. Use no seed for parallelism.
  warn(


UMAP ACC: 0.5397
UMAP ARI: 0.1947
Runtime (s): 9.2296
Embedding shape: (63, 6)


C:\Users\priyam\Python\Lib\site-packages\sklearn\cluster\_kmeans.py:1419: UserWarning: KMeans is known to have a memory leak on Windows with MKL, when there are less chunks than available threads. You can avoid it by setting the environment variable OMP_NUM_THREADS=1.
  warnings.warn(


### UMAP + K-Means Results

UMAP followed by K-Means achieved an ACC of 0.5397 and ARI of 0.1947 on SRBCT, compared with the paper's reported values of 0.543 and 0.190. The resulting embedding had six dimensions, and the execution time was approximately 10.12 seconds.

The results match my earlier experiment and are close to the published values, supporting the reproducibility of this baseline under the selected settings.

UMAP performed better than SpecGEM but did not outperform standardized K-Means in this experiment. This indicates that nonlinear dimensionality reduction does not necessarily guarantee improved clustering performance.

### UMAP Hyperparameter Sensitivity Analysis

I investigate how the number of nearest neighbors affects UMAP clustering performance on SRBCT.

The original paper uses `n_neighbors=8`. I compare this setting with 5 and 15 neighbors while keeping the embedding dimensionality, distance metric, and random seed unchanged.

This experiment examines the sensitivity of UMAP to a baseline hyperparameter. The original replication result is retained separately, and ground-truth labels are used only for evaluation.

In [13]:

import umap
from sklearn.cluster import KMeans
from sklearn.preprocessing import StandardScaler
import time

# Same preprocessing and PCA initialization as run_umap()
W = StandardScaler().fit_transform(X)
G = W @ W.T

eigenvalues, eigenvectors = np.linalg.eigh(G)
d = K + 2

init_embedding = eigenvectors[:, -d:][:, ::-1].copy()

umap_tuning_results = []

for neighbors in [5, 8, 15]:
    start = time.perf_counter()

    reducer = umap.UMAP(
        n_neighbors=neighbors,
        n_components=d,
        metric="cosine",
        angular_rp_forest=True,
        init=init_embedding.copy(),
        random_state=42
    )

    embedding = reducer.fit_transform(W)

    labels = KMeans(
        n_clusters=K,
        n_init=30,
        random_state=42
    ).fit_predict(embedding)

    runtime = time.perf_counter() - start
    metrics = evaluate_clustering(y, labels)

    umap_tuning_results.append({
        "n_neighbors": neighbors,
        "ACC": metrics["Accuracy"],
        "ARI": metrics["ARI"],
        "Runtime (s)": runtime
    })

umap_tuning_df = pd.DataFrame(umap_tuning_results)

display(umap_tuning_df.round(4))


C:\Users\priyam\Python\Lib\site-packages\umap\umap_.py:1952: UserWarning: n_jobs value 1 overridden to 1 by setting random_state. Use no seed for parallelism.
  warn(
C:\Users\priyam\Python\Lib\site-packages\sklearn\cluster\_kmeans.py:1419: UserWarning: KMeans is known to have a memory leak on Windows with MKL, when there are less chunks than available threads. You can avoid it by setting the environment variable OMP_NUM_THREADS=1.
  warnings.warn(
C:\Users\priyam\Python\Lib\site-packages\umap\umap_.py:1952: UserWarning: n_jobs value 1 overridden to 1 by setting random_state. Use no seed for parallelism.
  warn(
C:\Users\priyam\Python\Lib\site-packages\sklearn\cluster\_kmeans.py:1419: UserWarning: KMeans is known to have a memory leak on Windows with MKL, when there are less chunks than available threads. You can avoid it by setting the environment variable OMP_NUM_THREADS=1.
  warnings.warn(
C:\Users\priyam\Python\Lib\site-packages\umap\umap_.py:1952: UserWarning: n_jobs value 1 overr

,n_neighbors,ACC,ARI,Runtime (s)
0,5,0.5556,0.2008,0.1494
1,8,0.5397,0.1947,0.1629
2,15,0.5079,0.1659,0.1780


### UMAP Hyperparameter Sensitivity Results

I evaluated UMAP using three values of `n_neighbors` (5, 8, and 15), while keeping the other settings unchanged.

The original setting of 8 neighbors achieved an ACC of 0.5397 and ARI of 0.1947, which were close to the paper's reported values of 0.543 and 0.190. Reducing the number of neighbors to 5 improved ACC to 0.5556 and ARI to 0.2008, while increasing it to 15 reduced both metrics.

These results suggest that UMAP's clustering performance on SRBCT is sensitive to neighborhood size. Smaller neighborhoods performed better in this experiment, although additional runs would be needed to determine whether this pattern is consistent across random seeds and datasets.

## 5. IF-VAE Baseline

IF-VAE uses a variational autoencoder to learn a lower-dimensional representation of high-dimensional data, followed by clustering in the learned latent space.

I evaluate IF-VAE on SRBCT to compare a deep-learning-based representation with the iterative feature-selection methods. I will first reproduce the baseline using the paper's experimental settings and then examine selected hyperparameters separately.

In [14]:

import inspect
import Src.ifvae_compat as ifvae

# Inspect functions and their parameters
for name, func in inspect.getmembers(ifvae, inspect.isfunction):
    if func.__module__ == ifvae.__name__:
        print(name, inspect.signature(func))


feature_selection_ifvae (Data, per=1, seed=42, chunk_size=512)
run_ifvae (X, K, seed=42, per=1)
train_ifvae_embedding (X, latent_dim=25, batch_size=50, epochs=100, learning_rate=0.0005, seed=42)


In [15]:

import inspect
from Src.ifvae_compat import run_ifvae

print(inspect.getsource(run_ifvae))


def run_ifvae(X, K, seed=42, per=1):
    """
    IF-VAE baseline based on Chen, Jin & Ke (2023).

    X shape: (samples, features), matching data_loader.py.
    Returns predicted cluster labels and diagnostic details.
    """
    X = np.asarray(X, dtype=float)
    n, p = X.shape

    # Feature-wise sample standardization
    means = X.mean(axis=0)
    stds = X.std(axis=0, ddof=1)

    if np.any(stds == 0):
        raise ValueError("Constant features detected.")

    W = (X - means) / stds

    # Original IF-selection expects (features, samples)
    ranking, numselect = feature_selection_ifvae(
        pd.DataFrame(W.T), per=per, seed=seed
    )

    if numselect < 1:
        raise ValueError("Feature selection returned no features.")

    selected = ranking[:numselect]

    # VAE expects samples x selected features
    X_selected = W[:, selected]

    embedding = train_ifvae_embedding(
        X_selected,
        latent_dim=25,
        batch_size=50,
        epochs=100,
        learnin

In [16]:

import inspect
from Src.ifvae_compat import feature_selection_ifvae

print(inspect.getsource(feature_selection_ifvae))


def feature_selection_ifvae(Data, per=1, seed=42, chunk_size=512):
    """
    KS + Higher Criticism feature selection.

    Port of the feature_selection() procedure from
    Chen, Jin & Ke (2023).

    Data shape: (features, samples).
    """
    if not 0 < per <= 1:
        raise ValueError("per must be in (0, 1].")

    X = np.asarray(Data, dtype=float)
    p, n = X.shape

    if n < 3 or p < 4:
        raise ValueError("Insufficient data dimensions.")

    # Original feature-wise normalization
    means = X.mean(axis=1, keepdims=True)
    stds = X.std(axis=1, ddof=1, keepdims=True)

    if np.any(stds == 0):
        raise ValueError("Constant features detected.")

    W = (X - means) / stds

    # Simulate 50*p KS null statistics
    rep = 50 * p
    rng = np.random.RandomState(seed)
    KSvalue = np.empty(rep)

    kk_left = np.arange(n) / n
    kk_right = np.arange(1, n + 1) / n

    for start in range(0, rep, chunk_size):
        end = min(start + chunk_size, rep)
        z = r

In [17]:

from Src.ifvae_compat import run_ifvae
import time

# IF-VAE baseline using existing implementation settings
start_time = time.perf_counter()

ifvae_result = run_ifvae(
    X,
    K,
    seed=42,
    per=1
)

ifvae_runtime = time.perf_counter() - start_time

ifvae_metrics = evaluate_clustering(
    y,
    ifvae_result["labels"]
)

print("IF-VAE ACC:", round(ifvae_metrics["Accuracy"], 4))
print("IF-VAE ARI:", round(ifvae_metrics["ARI"], 4))
print("Selected features:", ifvae_result["n_selected"])
print("Embedding shape:", ifvae_result["embedding"].shape)
print("Runtime (s):", round(ifvae_runtime, 4))


IF-VAE ACC: 0.6349
IF-VAE ARI: 0.3017
Selected features: 51
Embedding shape: (63, 25)
Runtime (s): 10.8787


C:\Users\priyam\Python\Lib\site-packages\sklearn\cluster\_kmeans.py:1419: UserWarning: KMeans is known to have a memory leak on Windows with MKL, when there are less chunks than available threads. You can avoid it by setting the environment variable OMP_NUM_THREADS=1.
  warnings.warn(


IF-VAE Results

IF-VAE achieved an ACC of 0.6349 and ARI of 0.3017 on SRBCT, compared with the paper's reported values of 0.524 and 0.129. The method selected 51 genes and learned a 25-dimensional latent representation. The execution time was approximately 11.92 seconds.

In [18]:

import inspect
from Src.ifvae_compat import train_ifvae_embedding

print(inspect.getsource(train_ifvae_embedding))


def train_ifvae_embedding(
    X,
    latent_dim=25,
    batch_size=50,
    epochs=100,
    learning_rate=0.0005,
    seed=42
):
    """Train the VAE and return encoder means."""

    tf.keras.utils.set_random_seed(seed)

    X = np.asarray(X, dtype=np.float32)
    n, p = X.shape

    # Original 90/10 random split
    rng = np.random.RandomState(seed)
    n_test = round(0.1 * n)
    test_idx = rng.choice(n, n_test, replace=False)

    train_mask = np.ones(n, dtype=bool)
    train_mask[test_idx] = False

    X_train = X[train_mask]

    dataset = tf.data.Dataset.from_tensor_slices(X_train)
    dataset = dataset.shuffle(
        len(X_train), seed=seed, reshuffle_each_iteration=True
    ).batch(batch_size)

    model = IFVAEModel(p, latent_dim=latent_dim)

    # Build model before training
    model(tf.convert_to_tensor(X[:1]), training=False)

    optimizer = tf.keras.optimizers.Adam(
        learning_rate=learning_rate
    )

    beta = 0.0

    for epoch in range(epochs):
        for 

In [19]:

import inspect
from Src.ifvae_compat import IFVAEModel

print(inspect.getsource(IFVAEModel))


class IFVAEModel(tf.keras.Model):
    """
    Original VAE architecture:
    Dense -> BatchNorm -> ReLU for mean and log variance,
    reparameterization, followed by a sigmoid decoder.
    """

    def __init__(self, original_dim, latent_dim=25):
        super().__init__()

        self.mean_dense = tf.keras.layers.Dense(
            latent_dim, kernel_initializer="glorot_uniform"
        )
        self.mean_bn = tf.keras.layers.BatchNormalization()
        self.mean_relu = tf.keras.layers.ReLU()

        self.logvar_dense = tf.keras.layers.Dense(
            latent_dim, kernel_initializer="glorot_uniform"
        )
        self.logvar_bn = tf.keras.layers.BatchNormalization()
        self.logvar_relu = tf.keras.layers.ReLU()

        self.decoder = tf.keras.layers.Dense(
            original_dim,
            activation="sigmoid",
            kernel_initializer="glorot_uniform"
        )

    def encode(self, x, training=False):
        mu = self.mean_relu(
            self.mean_bn(se

## 6. Deep Embedded Clustering (DEC)

DEC is a deep-learning-based clustering method that learns a lower-dimensional representation using a neural network and refines cluster assignments through a clustering objective.

I evaluate DEC on SRBCT to compare deep clustering with the feature-selection and iterative refinement methods proposed in the i-IF-Learn paper.

I will first reproduce DEC using the paper's reported settings, then examine whether selected hyperparameters influence clustering performance.

In [20]:

import inspect
import Src.dec as dec

for name, func in inspect.getmembers(dec, inspect.isfunction):
    if func.__module__ == dec.__name__:
        print(name, inspect.signature(func))


pretrain_autoencoder (model, X_tensor, epochs=10, batch_size=256, learning_rate=0.001)
run_dec (X, K, seed=42)
target_distribution (q)
train_dec (model, clustering_layer, X_tensor, epochs=100, batch_size=256, learning_rate=0.001)


In [21]:

from Src.dec import run_dec
import inspect

print(inspect.getsource(run_dec))


def run_dec(X, K, seed=42):
    """
    Run the complete DEC pipeline:
    1. Standardize features
    2. Pretrain autoencoder
    3. Initialize cluster centers using KMeans
    4. Refine using KL divergence + reconstruction loss
    5. Return final cluster assignments
    """

    import random

    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)

    # Step 1: Standardize features
    X = np.asarray(X, dtype=np.float32)
    X_scaled = StandardScaler().fit_transform(X)
    X_tensor = torch.tensor(X_scaled, dtype=torch.float32)

    # Step 2: Initialize and pretrain autoencoder
    model = DECAutoencoder(input_dim=X.shape[1])

    pretrain_losses = pretrain_autoencoder(
        model,
        X_tensor,
        epochs=10,
        batch_size=256,
        learning_rate=0.001
    )

    # Step 3: Initialize clusters with KMeans
    model.eval()

    with torch.no_grad():
        initial_embedding = model.encoder(X_tensor).numpy()

    kmeans = KMeans(
        n_clust

In [22]:
from Src.dec import run_dec
import time

# DEC baseline using the original training settings
start_time = time.perf_counter()

dec_result = run_dec(
    X,
    K,
    seed=42
)

dec_runtime = time.perf_counter() - start_time

dec_metrics = evaluate_clustering(
    y,
    dec_result["labels"]
)

print("DEC ACC:", round(dec_metrics["Accuracy"], 4))
print("DEC ARI:", round(dec_metrics["ARI"], 4))
print("Runtime (s):", round(dec_runtime, 4))
print("Embedding shape:", dec_result["embedding"].shape)
print("Pretraining epochs:", len(dec_result["pretrain_losses"]))
print("DEC training epochs:", len(dec_result["history"]))


Pretraining Epoch 1/10, Loss: 1.009423
Pretraining Epoch 2/10, Loss: 1.050417
Pretraining Epoch 3/10, Loss: 0.986190
Pretraining Epoch 4/10, Loss: 0.960318
Pretraining Epoch 5/10, Loss: 0.913564
Pretraining Epoch 6/10, Loss: 0.852885
Pretraining Epoch 7/10, Loss: 0.794572
Pretraining Epoch 8/10, Loss: 0.742245
Pretraining Epoch 9/10, Loss: 0.694860
Pretraining Epoch 10/10, Loss: 0.652401


C:\Users\priyam\Python\Lib\site-packages\sklearn\cluster\_kmeans.py:1419: UserWarning: KMeans is known to have a memory leak on Windows with MKL, when there are less chunks than available threads. You can avoid it by setting the environment variable OMP_NUM_THREADS=1.
  warnings.warn(


DEC Epoch 10/100 | Reconstruction: 0.554570 | KL: 0.105846 | Total: 0.660416
DEC Epoch 20/100 | Reconstruction: 0.460309 | KL: 0.103121 | Total: 0.563430
DEC Epoch 30/100 | Reconstruction: 0.401967 | KL: 0.102904 | Total: 0.504871
DEC Epoch 40/100 | Reconstruction: 0.358308 | KL: 0.101555 | Total: 0.459863
DEC Epoch 50/100 | Reconstruction: 0.320992 | KL: 0.099581 | Total: 0.420573
DEC Epoch 60/100 | Reconstruction: 0.294384 | KL: 0.098430 | Total: 0.392814
DEC Epoch 70/100 | Reconstruction: 0.270669 | KL: 0.096653 | Total: 0.367322
DEC Epoch 80/100 | Reconstruction: 0.248336 | KL: 0.095081 | Total: 0.343416
DEC Epoch 90/100 | Reconstruction: 0.229095 | KL: 0.094565 | Total: 0.323660
DEC Epoch 100/100 | Reconstruction: 0.221792 | KL: 0.093881 | Total: 0.315673
DEC ACC: 0.5397
DEC ARI: 0.1805
Runtime (s): 4.2823
Embedding shape: (63, 10)
Pretraining epochs: 10
DEC training epochs: 3


DEC Results

DEC achieved an ACC of 0.5397 and ARI of 0.1805 on SRBCT, compared with the paper's reported values of 0.460 and 0.082. The execution time was approximately 4.28 seconds, and the learned embedding contained 10 dimensions.

During training, the autoencoder reconstruction loss generally decreased, and the DEC refinement loss also decreased over 100 epochs. This indicates that the model was optimizing its training objective, although lower training loss does not necessarily correspond to improved clustering performance.

In [23]:

print("History type:", type(dec_result["history"]))

if isinstance(dec_result["history"], dict):
    print("History keys:", list(dec_result["history"].keys()))
else:
    print("History entries:", dec_result["history"])


History type: <class 'dict'>
History keys: ['reconstruction_loss', 'kl_loss', 'total_loss']


## 7. DeepCluster Baseline

DeepCluster is a deep-learning-based clustering method that alternates between generating pseudo-labels through K-Means and updating a neural network using those pseudo-labels.

I evaluate DeepCluster on SRBCT to compare its iterative clustering approach with i-IF-Learn. Unlike i-IF-Learn, which iteratively selects informative features, DeepCluster focuses on learning a representation through neural-network training.

The paper uses hyperparameter optimization for DeepCluster, so I will first inspect the existing implementation and its settings before running the baseline.

In [24]:

import inspect
import Src.deepcluster as deepcluster

for name, func in inspect.getmembers(
    deepcluster, inspect.isfunction
):
    if func.__module__ == deepcluster.__name__:
        print(name, inspect.signature(func))


generate_pseudo_labels (embedding, n_clusters, seed)
get_embedding (model, X_tensor)
run_deepcluster (X, K, seed=42, n_trials=10, latent_dim=10, batch_size=64)
set_seed (seed=42)
train_deepcluster (X, n_clusters, hidden_dim=128, latent_dim=10, epochs=10, iterations=5, learning_rate=0.001, batch_size=64, seed=42)


In [25]:

import inspect
from Src.deepcluster import run_deepcluster

print(inspect.getsource(run_deepcluster))


def run_deepcluster(
    X,
    K,
    seed=42,
    n_trials=10,
    latent_dim=10,
    batch_size=64
):

    X = np.asarray(X, dtype=np.float32)

    if X.ndim != 2:
        raise ValueError("X must be a 2D data matrix.")

    if not 2 <= K < len(X):
        raise ValueError("K must be between 2 and n_samples - 1.")

    # Standardize all features
    scaler = StandardScaler()
    X_scaled = scaler.fit_transform(X).astype(np.float32)

    def objective(trial):

        hidden_dim = trial.suggest_categorical(
            "hidden_dim", [64, 128, 256]
        )

        epochs = trial.suggest_int(
            "epochs", 5, 15
        )

        iterations = trial.suggest_int(
            "iterations", 3, 10
        )

        result = train_deepcluster(
            X=X_scaled,
            n_clusters=K,
            hidden_dim=hidden_dim,
            latent_dim=latent_dim,
            epochs=epochs,
            iterations=iterations,
            learning_rate=0.001,
            batch_size=b

In [26]:

from Src.deepcluster import run_deepcluster
import time

# DeepCluster baseline with Optuna hyperparameter optimization

start_time = time.perf_counter()

deepcluster_result = run_deepcluster(
    X,
    K,
    seed=42,
    n_trials=10,
    latent_dim=10,
    batch_size=64
)

deepcluster_runtime = time.perf_counter() - start_time

deepcluster_metrics = evaluate_clustering(
    y,
    deepcluster_result["labels"]
)

print("DeepCluster ACC:", round(deepcluster_metrics["Accuracy"], 4))
print("DeepCluster ARI:", round(deepcluster_metrics["ARI"], 4))
print("Runtime (s):", round(deepcluster_runtime, 4))
print("Best hyperparameters:", deepcluster_result["best_params"])
print("Best Optuna score:", round(deepcluster_result["best_optuna_score"], 4))


[I 2026-11-04 23:51:39,507] A new study created in memory with name: no-name-6d4c73a3-2c8d-49e3-84ff-80bb1bb8f686
C:\Users\priyam\Python\Lib\site-packages\sklearn\cluster\_kmeans.py:1419: UserWarning: KMeans is known to have a memory leak on Windows with MKL, when there are less chunks than available threads. You can avoid it by setting the environment variable OMP_NUM_THREADS=1.
  warnings.warn(
C:\Users\priyam\Python\Lib\site-packages\sklearn\cluster\_kmeans.py:1419: UserWarning: KMeans is known to have a memory leak on Windows with MKL, when there are less chunks than available threads. You can avoid it by setting the environment variable OMP_NUM_THREADS=1.
  warnings.warn(
C:\Users\priyam\Python\Lib\site-packages\sklearn\cluster\_kmeans.py:1419: UserWarning: KMeans is known to have a memory leak on Windows with MKL, when there are less chunks than available threads. You can avoid it by setting the environment variable OMP_NUM_THREADS=1.
  warnings.warn(
C:\Users\priyam\Python\Lib\s

DeepCluster ACC: 0.5714
DeepCluster ARI: 0.2282
Runtime (s): 12.7862
Best hyperparameters: {'hidden_dim': 64, 'epochs': 15, 'iterations': 10}
Best Optuna score: 0.2031


C:\Users\priyam\Python\Lib\site-packages\sklearn\cluster\_kmeans.py:1419: UserWarning: KMeans is known to have a memory leak on Windows with MKL, when there are less chunks than available threads. You can avoid it by setting the environment variable OMP_NUM_THREADS=1.
  warnings.warn(


DeepCluster Results

DeepCluster achieved an ACC of 0.5714 and ARI of 0.2282 on SRBCT, compared with the paper's reported values of 0.546 and 0.162. The execution time was approximately 12.79 seconds.

Optuna selected a hidden layer size of 64, 15 epochs per clustering iteration, and 10 clustering-training iterations. Hyperparameters were selected using an unsupervised objective based on silhouette score and training loss.